# 00 — Colab G4 preflight

## Goal

Prove the G4 runtime, pinned day-zero package set, Hub
authentication, BF16 model load and native Qwen3.8 tool template
before any training. This notebook performs no optimisation.

**Gate:** do not continue if the GPU, model load, template or
durable manifest check fails.

## Setup

Select the **G4** runtime and add a write-capable `HF_TOKEN` in
Colab Secrets. Run the install cell once, restart the runtime,
then rerun the notebook from the top. The marker makes the
install cell cheap and idempotent on the second pass.

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

In [ ]:
import gc
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# IPython stores the last exception on sys.last_traceback, whose frames keep
# every local alive, including a ~52 GiB model from a failed cell. gc.collect()
# cannot free what those frames still reference.
def release_stale_gpu_state() -> float:
    for _stale_name in ("model", "tokenizer", "processor", "trainer"):
        globals().pop(_stale_name, None)
    for _exc_attr in ("last_traceback", "last_value", "last_type", "last_exc"):
        if hasattr(sys, _exc_attr):
            delattr(sys, _exc_attr)
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch._dynamo.reset()
    except AttributeError:
        pass
    return torch.cuda.mem_get_info()[0] / 1024**3

# Fail before a model load that accelerate would silently offload.
def require_free_vram(minimum_gib: float) -> float:
    free_gib = release_stale_gpu_state()
    if free_gib < minimum_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free but this load needs about "
            f"{minimum_gib:.0f} GiB. A previous model in this kernel is still "
            "holding memory. Restart the runtime and rerun from the top."
        )
    return free_gib

# Reject a load that accelerate quietly spilled to CPU or disk. A partially
# offloaded model copies weights back per forward pass (the 2.4 GiB embedding
# alone) and is guaranteed to OOM or crawl mid-episode.
def assert_model_fully_resident(model, minimum_free_gib: float = 4.0) -> None:
    non_cuda = sorted({
        parameter.device.type
        for parameter in model.parameters()
        if parameter.device.type != "cuda"
    })
    offload_hooks = [
        name for name, module in model.named_modules()
        if getattr(getattr(module, "_hf_hook", None), "offload", False)
    ]
    if non_cuda or offload_hooks:
        raise RuntimeError(
            "The checkpoint did not fit on the GPU and accelerate offloaded "
            f"part of it (devices={non_cuda}, offload_hooks={len(offload_hooks)}). "
            "Restart the runtime to release stale VRAM, then rerun from the top."
        )
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    if free_gib < minimum_free_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free after the load; the KV "
            "cache and generation workspaces need headroom. Restart the "
            "runtime and rerun from the top."
        )
    print(f"Model fully resident on GPU; {free_gib:.1f} GiB VRAM free.")

release_stale_gpu_state()

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Load the trainable BF16 checkpoint

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen3.8-27B"
MODEL_REVISION = None  # Set to an immutable Hub commit after the first successful load.
MAX_SEQUENCE_LENGTH = 4096

load_kwargs = {
    "model_name": MODEL_ID,
    "max_seq_length": MAX_SEQUENCE_LENGTH,
    "load_in_4bit": False,
    "load_in_8bit": False,
    "full_finetuning": False,
}
if MODEL_REVISION:
    load_kwargs["revision"] = MODEL_REVISION

require_free_vram(60.0)
torch.cuda.reset_peak_memory_stats()
model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
assert_model_fully_resident(model)
peak_gib = torch.cuda.max_memory_reserved() / 1024**3
print(f"Loaded {MODEL_ID}; peak reserved VRAM={peak_gib:.2f} GiB")

model_type = getattr(model.config, "model_type", None)
text_config = getattr(model.config, "text_config", model.config)
assert model_type == "qwen3_5", model_type
assert getattr(text_config, "max_position_embeddings", None) == 262144

## Validate the native tool template

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a restricted allow-listed command. It is disabled in the pilot.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Fold an initial developer message into system for the HF tokenizer.

    The adapter performs the same mapping in training and deployment. The
    official safetensor tokenizer currently accepts system/user/assistant/tool.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool) -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort="medium",
        preserve_thinking=True,
    )

In [ ]:
template_probe = [
    {"role": "developer", "content": "Work only in the provided repository and verify changes."},
    {"role": "user", "content": "Read src/cache.py before proposing a fix."},
]
rendered_probe = render_chat(template_probe, add_generation_prompt=True)
assert rendered_tool_schema(rendered_probe) == TOOL_SCHEMA_JSON, (
    "The tokenizer changed the deployment tool declarations."
)
assert "Work only in the provided repository" in rendered_probe, (
    "The developer-to-system adapter lost the repository policy."
)
assert rendered_probe.endswith("<think>\n"), (
    "The generation prompt no longer opens Qwen's thinking channel."
)
print(rendered_probe[:4000])

## Run one bounded inference probe

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    text=rendered_probe,
    return_tensors="pt",
    add_special_tokens=False,
).to("cuda")
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
        do_sample=True,
        use_cache=True,
    )
new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
generated = tokenizer.decode(new_tokens, skip_special_tokens=False)
print(generated)
print(f"Peak reserved VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GiB")

## Inspect language and vision parameters

In [ ]:
vision_markers = ("vision", "visual", "image")
vision_names = [
    name for name, _ in model.named_parameters()
    if any(marker in name.lower() for marker in vision_markers)
]
linear_suffixes = sorted({
    name.rsplit(".", 1)[-1]
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and not any(marker in name.lower() for marker in vision_markers)
})
print(f"Vision-associated parameters: {len(vision_names)}")
print("Language linear suffixes:", linear_suffixes)
assert vision_names, "Expected the multimodal checkpoint to expose vision parameters."

## Checks and next step

The preflight passes when the model loads in BF16, the rendered
prompt contains the exact XML tool syntax, a bounded generation
completes, and `runtime_manifest.json` exists. Record the Hub
commit used, then continue to `01_tool_calling_baseline.ipynb`.